# Assignment 2: Advanced RAG Techniques
## Day 6 Session 2 - Advanced RAG Fundamentals

**OBJECTIVE:** Implement advanced RAG techniques including postprocessors, response synthesizers, and structured outputs.

**LEARNING GOALS:**
- Understand and implement node postprocessors for filtering and reranking
- Learn different response synthesis strategies (TreeSummarize, Refine)
- Create structured outputs using Pydantic models
- Build advanced retrieval pipelines with multiple processing stages

**DATASET:** Use the same data folder as Assignment 1 (`Day_6/session_2/data/`)

**PREREQUISITES:** Complete Assignment 1 first

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it
3. The answers can be found in the `03_advanced_rag_techniques.ipynb` notebook
4. Each technique builds on the previous one


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -r /content/drive/MyDrive/ai-accelerator-C5-main/Day_6/session_2/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 13.7 MB/s eta 0:00:00
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
INFO: pip is looking at multiple versions of llama-index-llms-openai-like to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-index-llms-openai-like to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of llama-index-

In [1]:
import os
from pathlib import Path
from typing import Dict, List, Optional, Any
from pydantic import BaseModel, Field

# Core LlamaIndex components
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever
print("✅ LlamaIndex core components imported successfully!")

# Vector store
from llama_index.vector_stores.lancedb import LanceDBVectorStore
print("✅ LanceDBVectorStore imported successfully!")

# Embeddings and LLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
print("✅ HuggingFaceEmbedding imported successfully!")
from llama_index.llms.openrouter import OpenRouter
print("✅ OpenRouter LLM imported successfully!")

# Advanced RAG components (we'll use these in the assignments)
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.response_synthesizers import TreeSummarize, Refine, CompactAndRefine
from llama_index.core.output_parsers import PydanticOutputParser
print("✅ Advanced RAG specific components imported successfully!")

print("✅ Advanced RAG libraries imported successfully!")

✅ LlamaIndex core components imported successfully!
✅ LanceDBVectorStore imported successfully!
✅ HuggingFaceEmbedding imported successfully!
✅ OpenRouter LLM imported successfully!
✅ Advanced RAG specific components imported successfully!
✅ Advanced RAG libraries imported successfully!


In [5]:
# Configure Advanced RAG Settings (Using OpenRouter)
from google.colab import userdata

def setup_advanced_rag_settings():
    """
    Configure LlamaIndex with optimized settings for advanced RAG.
    Uses local embeddings and OpenRouter for LLM operations.
    """
    # Check for OpenRouter API key
    #api_key = os.getenv("OPENROUTER_API_KEY")
    api_key = userdata.get('OPENROUTER_API_KEY')
    if not api_key:
        print("⚠️  OPENROUTER_API_KEY not found - LLM operations will be limited")
        print("   You can still complete postprocessor and retrieval exercises")
    else:
        print("✅ OPENROUTER_API_KEY found - full advanced RAG functionality available")

        # Configure OpenRouter LLM
        Settings.llm = OpenRouter(
            api_key=api_key,
            model="gpt-4o",
            temperature=0.1  # Lower temperature for more consistent responses
        )

    # Configure local embeddings (no API key required)
    Settings.embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        trust_remote_code=True
    )

    # Advanced RAG configuration
    Settings.chunk_size = 512  # Smaller chunks for better precision
    Settings.chunk_overlap = 50

    print("✅ Advanced RAG settings configured")
    print("   - Chunk size: 512 (optimized for precision)")
    print("   - Using local embeddings for cost efficiency")
    print("   - OpenRouter LLM ready for response synthesis")

# Setup the configuration
setup_advanced_rag_settings()


✅ OPENROUTER_API_KEY found - full advanced RAG functionality available
✅ Advanced RAG settings configured
   - Chunk size: 512 (optimized for precision)
   - Using local embeddings for cost efficiency
   - OpenRouter LLM ready for response synthesis


In [7]:
# Setup: Create index from Assignment 1 (reuse the basic functionality)
def setup_basic_index(data_folder: str = "../data", force_rebuild: bool = False):
    """
    Create a basic vector index that we'll enhance with advanced techniques.
    This reuses the concepts from Assignment 1.

    Args:
        data_folder (str): The path to the directory containing the documents.
        force_rebuild (bool): If True, forces a rebuild of the index even if it exists.
                              (Currently not fully implemented for force_rebuild logic,
                              but can be extended to clear existing LanceDB data).

    Returns:
        VectorStoreIndex: An initialized LlamaIndex VectorStoreIndex, or None if setup fails.
    """
    print("\n--- Starting Basic Index Setup ---")
    print(f"🔄 Attempting to set up basic index from data folder: {data_folder}")

    # Step 1: Initialize the Vector Store
    # We're using LanceDB as our vector database. LanceDB is an open-source, column-oriented
    # database for vector embeddings, designed for speed and efficiency.
    # `uri` specifies where the database files will be stored locally.
    # `table_name` is the name of the table within the LanceDB database where our vectors will reside.
    print("✨ Initializing LanceDB Vector Store...")
    vector_store = LanceDBVectorStore(
        uri="./advanced_rag_vectordb",
        table_name="documents"
    )
    print("✅ LanceDB Vector Store initialized at './advanced_rag_vectordb' with table 'documents'.")

    # Step 2: Load documents from the specified data folder
    # The SimpleDirectoryReader is a utility from LlamaIndex that helps load various file types
    # (like .txt, .pdf, .md, etc.) from a directory into LlamaIndex Document objects.
    # `recursive=True` ensures that it will look into subfolders as well.
    print(f"📁 Checking if data folder '{data_folder}' exists...")
    if not Path(data_folder).exists():
        print(f"❌ Error: Data folder not found at '{data_folder}'. Please ensure the path is correct.")
        return None
    print(f"✅ Data folder '{data_folder}' found. Proceeding to load documents.")

    print("📚 Loading documents using SimpleDirectoryReader (this might take a moment if many files)...")
    reader = SimpleDirectoryReader(input_dir=data_folder, recursive=True)
    documents = reader.load_data()
    print(f"✅ Successfully loaded {len(documents)} documents from the data folder.")

    if not documents:
        print("⚠️ No documents were loaded. Index will not be created. Check your data folder contents.")
        return None

    # Step 3: Create Storage Context
    # The StorageContext manages how and where the index stores its data (nodes, vectors, etc.).
    # We pass our pre-configured `vector_store` (LanceDB) to it, so LlamaIndex knows where to
    # save the document embeddings.
    print("📦 Creating Storage Context with LanceDB Vector Store...")
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    print("✅ Storage Context created.")

    # Step 4: Create the VectorStoreIndex
    # This is the core step where the magic happens! LlamaIndex takes our loaded `documents`,
    # converts them into 'nodes' (chunks of text), generates numerical embeddings (vectors) for each node
    # using the `Settings.embed_model` (configured earlier in `setup_advanced_rag_settings`),
    # and then stores these vectors in our `storage_context` (LanceDB).
    # `show_progress=True` provides visual feedback during the embedding process.
    print("🧠 Building VectorStoreIndex from documents (this will generate embeddings and store them)...")
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )
    print("✅ VectorStoreIndex creation complete!")

    print(f"✅ Basic index created with {len(documents)} documents processed.")
    print("   Ready for advanced RAG techniques!")
    print("--- Basic Index Setup Finished ---\n")
    return index

# --- Execution of the setup function ---
print("📁 Setting up basic index for advanced RAG...")
# Call the function to create our initial index.
# Make sure the data folder path is correct relative to the notebook or an absolute path.
index = setup_basic_index(data_folder="/content/drive/MyDrive/ai-accelerator-C5-main/Day_6/session_2/data")

# Final check to confirm index creation status
if index:
    print("🚀 Index successfully created! You are ready to implement advanced RAG techniques.")
else:
    print("❌ Failed to create index. Please review the output above for errors and ensure the data folder path is correct.")

📁 Setting up basic index for advanced RAG...

--- Starting Basic Index Setup ---
🔄 Attempting to set up basic index from data folder: /content/drive/MyDrive/ai-accelerator-C5-main/Day_6/session_2/data
✨ Initializing LanceDB Vector Store...
✅ LanceDB Vector Store initialized at './advanced_rag_vectordb' with table 'documents'.
📁 Checking if data folder '/content/drive/MyDrive/ai-accelerator-C5-main/Day_6/session_2/data' exists...
✅ Data folder '/content/drive/MyDrive/ai-accelerator-C5-main/Day_6/session_2/data' found. Proceeding to load documents.
📚 Loading documents using SimpleDirectoryReader (this might take a moment if many files)...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Successfully loaded 42 documents from the data folder.
📦 Creating Storage Context with LanceDB Vector Store...
✅ Storage Context created.
🧠 Building VectorStoreIndex from documents (this will generate embeddings and store them)...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/93 [00:00<?, ?it/s]

✅ VectorStoreIndex creation complete!
✅ Basic index created with 42 documents processed.
   Ready for advanced RAG techniques!
--- Basic Index Setup Finished ---

🚀 Index successfully created! You are ready to implement advanced RAG techniques.


## 1. Node Postprocessors - Similarity Filtering

**Concept:** Postprocessors refine retrieval results after the initial vector search. The `SimilarityPostprocessor` filters out chunks that fall below a relevance threshold.

**Why it matters:** Raw vector search often returns some irrelevant results. Filtering improves precision and response quality.

Complete the function below to create a query engine with similarity filtering.


In [8]:
def create_query_engine_with_similarity_filter(index, similarity_cutoff: float = 0.3, top_k: int = 10):
    """
    Create a query engine that filters results based on similarity scores.

    Args:
        index: Vector index to query
        similarity_cutoff: Minimum similarity score (0.0 to 1.0)
        top_k: Number of initial results to retrieve before filtering

    Returns:
        Query engine with similarity filtering
    """
    print(f"\n--- Starting Query Engine with Similarity Filter Setup ---")
    print(f"⚙️ Configuring query engine with similarity_cutoff={similarity_cutoff} and top_k={top_k}.")

    # Step 1: Initialize the SimilarityPostprocessor
    # The SimilarityPostprocessor is used to filter out nodes (document chunks)
    # that have a similarity score below a certain threshold. This helps ensure
    # that only the most relevant information is considered for response synthesis,
    # improving the precision of the RAG system.
    print(f"✨ Initializing SimilarityPostprocessor with a cutoff threshold of {similarity_cutoff}...")
    similarity_processor = SimilarityPostprocessor(similarity_cutoff=similarity_cutoff)
    print(f"✅ SimilarityPostprocessor created.")

    # Step 2: Create the Query Engine
    # We create a query engine from the existing vector index. The key here is
    # passing the `node_postprocessors` parameter, which tells LlamaIndex to apply
    # our `similarity_processor` after the initial retrieval step.
    # The `similarity_top_k` in `as_query_engine` determines how many nodes are
    # retrieved *before* the postprocessor filters them. A higher `top_k` allows
    # the postprocessor to choose from a wider pool of initial candidates.
    print(f"🧠 Creating RetrieverQueryEngine with {top_k} initial results and applying similarity filtering...")
    query_engine = index.as_query_engine(
        similarity_top_k=top_k, # Retrieve 'top_k' nodes initially
        node_postprocessors=[similarity_processor] # Apply the similarity filter
    )
    print("✅ Query engine with similarity filtering created.")
    print("--- Query Engine with Similarity Filter Setup Finished ---\n")
    return query_engine

# Test the function
if index:
    filtered_engine = create_query_engine_with_similarity_filter(index, similarity_cutoff=0.3)

    if filtered_engine:
        print("✅ Query engine with similarity filtering created")

        # Test query
        test_query = "What are the benefits of AI agents?"
        print(f"\n🔍 Testing query: '{test_query}'")
        print("   Querying the filtered engine...")

        # Uncomment when implemented:
        response = filtered_engine.query(test_query)
        print(f"📝 Response received from filtered engine: ")
        print(str(response))
        # print("   (Complete the function above to test the response)")
    else:
        print("❌ Failed to create filtered query engine")
else:
    print("❌ No index available - run previous cells first")


--- Starting Query Engine with Similarity Filter Setup ---
⚙️ Configuring query engine with similarity_cutoff=0.3 and top_k=10.
✨ Initializing SimilarityPostprocessor with a cutoff threshold of 0.3...
✅ SimilarityPostprocessor created.
🧠 Creating RetrieverQueryEngine with 10 initial results and applying similarity filtering...
✅ Query engine with similarity filtering created.
--- Query Engine with Similarity Filter Setup Finished ---

✅ Query engine with similarity filtering created

🔍 Testing query: 'What are the benefits of AI agents?'
   Querying the filtered engine...
📝 Response received from filtered engine: 
AI agents offer several benefits, including enhanced reasoning and planning capabilities that facilitate effective interaction with complex environments and autonomous decision-making. They can execute tasks iteratively, using tools to interact with external data sources and adjust plans based on new feedback or information. Multi-agent systems, in particular, provide clear 

## 2. Response Synthesizers - TreeSummarize

**Concept:** Response synthesizers control how retrieved information becomes final answers. `TreeSummarize` builds responses hierarchically, ideal for complex analytical questions.

**Why it matters:** Different synthesis strategies work better for different query types. TreeSummarize excels at comprehensive analysis and long-form responses.

Complete the function below to create a query engine with TreeSummarize response synthesis.


In [9]:
def create_query_engine_with_tree_summarize(index, top_k: int = 5):
    """
    Create a query engine that uses TreeSummarize for comprehensive responses.

    Args:
        index: Vector index to query
        top_k: Number of results to retrieve for initial context.

    Returns:
        Query engine with TreeSummarize synthesis
    """
    print(f"\n--- Starting Query Engine with TreeSummarize Setup ---")
    print(f"⚙️ Configuring query engine for TreeSummarize synthesis with top_k={top_k}.")

    # Step 1: Initialize the TreeSummarize response synthesizer
    # TreeSummarize is a response synthesis strategy that works by recursively
    # summarizing retrieved nodes until a single root summary node is obtained.
    # This is particularly useful for complex, analytical questions requiring
    # a comprehensive answer derived from multiple pieces of information.
    print(f"✨ Initializing TreeSummarize response synthesizer...")
    tree_synthesizer = TreeSummarize()
    print(f"✅ TreeSummarize response synthesizer created.")

    # Step 2: Create the Query Engine with the TreeSummarize synthesizer
    # When calling `index.as_query_engine()`, we can specify the `response_synthesizer`
    # parameter to use a custom synthesis strategy. Here, we're passing our
    # `tree_synthesizer` instance.
    # `similarity_top_k` controls how many initial nodes are retrieved from the index
    # before the response synthesizer begins its work.
    print(f"🧠 Creating RetrieverQueryEngine with top_k={top_k} and TreeSummarize synthesis...")
    query_engine = index.as_query_engine(
        similarity_top_k=top_k,
        response_synthesizer=tree_synthesizer
    )
    print("✅ Query engine with TreeSummarize synthesis created.")
    print("--- Query Engine with TreeSummarize Setup Finished ---\n")
    return query_engine

# Test the function
if index:
    tree_engine = create_query_engine_with_tree_summarize(index)

    if tree_engine:
        print("✅ Query engine with TreeSummarize created")

        # Test with a complex analytical query
        analytical_query = "Compare the advantages and disadvantages of different AI agent frameworks"
        print(f"\n🔍 Testing analytical query: '{analytical_query}'")
        print("   Querying the TreeSummarize engine...")

        # Uncomment when implemented:
        response = tree_engine.query(analytical_query)
        print(f"📝 TreeSummarize Response:\n")
        print(str(response))
    else:
        print("❌ Failed to create TreeSummarize query engine")
else:
    print("❌ No index available - run previous cells first")


--- Starting Query Engine with TreeSummarize Setup ---
⚙️ Configuring query engine for TreeSummarize synthesis with top_k=5.
✨ Initializing TreeSummarize response synthesizer...
✅ TreeSummarize response synthesizer created.
🧠 Creating RetrieverQueryEngine with top_k=5 and TreeSummarize synthesis...
✅ Query engine with TreeSummarize synthesis created.
--- Query Engine with TreeSummarize Setup Finished ---

✅ Query engine with TreeSummarize created

🔍 Testing analytical query: 'Compare the advantages and disadvantages of different AI agent frameworks'
   Querying the TreeSummarize engine...
📝 TreeSummarize Response:

Different AI agent frameworks offer various advantages and disadvantages based on their design and intended use cases.

**Advantages:**

1. **Autonomous Agents:**
   - **AutoGPT**: Known for its pioneering role in autonomous task execution, making it suitable for complex, independent tasks.
   - **BabyAGI**: Offers a simplified approach, which can be beneficial for those lo

## 3. Structured Outputs with Pydantic Models

**Concept:** Structured outputs ensure predictable, parseable responses using Pydantic models. This is essential for API endpoints and data pipelines.

**Why it matters:** Instead of free-text responses, you get type-safe, validated data structures that applications can reliably process.

Complete the function below to create a structured output system for extracting research paper information.


In [12]:
# First, define the Pydantic models for structured outputs
class ResearchPaperInfo(BaseModel):
    """Structured information about a research paper or AI concept."""
    title: str = Field(description="The main title or concept name")
    key_points: List[str] = Field(description="3-5 main points or findings")
    applications: List[str] = Field(description="Practical applications or use cases")
    summary: str = Field(description="Brief 2-3 sentence summary")

# Import the missing component
from llama_index.core.program import LLMTextCompletionProgram

def create_structured_output_program(output_model: BaseModel = ResearchPaperInfo):
    """
    Create a structured output program using Pydantic models.

    Args:
        output_model: Pydantic model class for structured output

    Returns:
        LLMTextCompletionProgram that returns structured data
    """
    # Create output parser with the Pydantic model
    output_parser = PydanticOutputParser(output_model)

    # Define a prompt template for structured extraction
    prompt_template_str = (
        "Given the following context, please extract the requested information.\n"
        "Context: {context}\n"
        "Query: {query}\n"
        "Output:\n"
    )

    # Create the structured output program
    program = LLMTextCompletionProgram.from_defaults(
        output_parser=output_parser,
        prompt_template_str=prompt_template_str,
        llm=Settings.llm # Use the globally configured LLM
    )

    return program

# Test the function
if index:
    structured_program = create_structured_output_program(ResearchPaperInfo)

    if structured_program:
        print("✅ Structured output program created")

        # Test with retrieval and structured extraction
        structure_query = "Tell me about AI agents and their capabilities"
        print(f"\n🔍 Testing structured query: '{structure_query}'")

        # Get context for structured extraction
        retriever = VectorIndexRetriever(index=index, similarity_top_k=3)
        nodes = retriever.retrieve(structure_query)
        context = "\n".join([node.text for node in nodes])

        # Get structured response
        response = structured_program(context=context, query=structure_query)
        print(f"📊 Structured Response:\n{response.json()}") # Use .json() for pretty printing the Pydantic object

        print("\n💡 Expected output format:")
        print("   - title: String")
        print("   - key_points: List of strings")
        print("   - applications: List of strings")
        print("   - summary: String")
    else:
        print("❌ Failed to create structured output program")
else:
    print("❌ No index available - run previous cells first")


✅ Structured output program created

🔍 Testing structured query: 'Tell me about AI agents and their capabilities'
📊 Structured Response:
{"title":"AI Agents and Their Capabilities","key_points":["AI agents enhance language model capabilities for solving real-world challenges.","Single and multi-agent architectures exhibit strong performance on complex tasks.","Challenges exist in agent evaluation due to varying benchmarks and potential biases.","Dynamic teams and intelligent message filtering improve multi-agent performance.","Vertical and horizontal architectures offer different collaboration and communication structures."],"applications":["Real-world problem-solving through reasoning and planning.","Tool execution and interaction with external environments.","Collaboration on complex tasks requiring multiple agents."],"summary":"AI agents extend language model capabilities to address real-world challenges by employing reasoning, planning, and tool execution. Both single and multi-age

/tmp/ipykernel_5970/1971105483.py:60: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"📊 Structured Response:\n{response.json()}") # Use .json() for pretty printing the Pydantic object


## 4. Advanced Pipeline - Combining All Techniques

**Concept:** Combine multiple advanced techniques into a single powerful query engine: similarity filtering + response synthesis + structured output.

**Why it matters:** Production RAG systems often need multiple techniques working together for optimal results.

Complete the function below to create a comprehensive advanced RAG pipeline.


In [13]:
def create_advanced_rag_pipeline(index, similarity_cutoff: float = 0.3, top_k: int = 10):
    """
    Create a comprehensive advanced RAG pipeline combining multiple techniques.

    Args:
        index: Vector index to query
        similarity_cutoff: Minimum similarity score for filtering
        top_k: Number of initial results to retrieve

    Returns:
        Advanced query engine with filtering and synthesis combined
    """
    print(f"\n--- Starting Advanced RAG Pipeline Setup ---")
    print(f"⚙️ Configuring advanced pipeline with similarity_cutoff={similarity_cutoff} and top_k={top_k}.")

    # Step 1: Create similarity postprocessor
    # This postprocessor will filter out nodes that have a similarity score below the specified cutoff.
    print(f"✨ Initializing SimilarityPostprocessor with a cutoff of {similarity_cutoff}...")
    similarity_processor = SimilarityPostprocessor(similarity_cutoff=similarity_cutoff)
    print("✅ SimilarityPostprocessor created.")

    # Step 2: Create TreeSummarize for comprehensive responses
    # TreeSummarize is a response synthesis strategy for analytical and comprehensive answers.
    print("✨ Initializing TreeSummarize response synthesizer...")
    tree_synthesizer = TreeSummarize()
    print("✅ TreeSummarize response synthesizer created.")

    # Step 3: Create the comprehensive query engine combining both techniques
    # The query engine will first retrieve 'top_k' nodes, then apply the similarity filter,
    # and finally use TreeSummarize to synthesize a comprehensive response.
    print("🧠 Creating RetrieverQueryEngine with SimilarityPostprocessor and TreeSummarize synthesis...")
    advanced_engine = index.as_query_engine(
        similarity_top_k=top_k,
        node_postprocessors=[similarity_processor],
        response_synthesizer=tree_synthesizer
    )
    print("✅ Advanced RAG pipeline (query engine) created.")
    print("--- Advanced RAG Pipeline Setup Finished ---\n")

    return advanced_engine

# Test the comprehensive pipeline
if index:
    advanced_pipeline = create_advanced_rag_pipeline(index)

    if advanced_pipeline:
        print("✅ Advanced RAG pipeline created successfully!")
        print("   🔧 Similarity filtering: ✅")
        print("   🌳 TreeSummarize synthesis: ✅")

        # Test with complex query
        complex_query = "Analyze the current state and future potential of AI agent technologies"
        print(f"\n🔍 Testing complex query: '{complex_query}'")

        # Uncomment when implemented:
        response = advanced_pipeline.query(complex_query)
        print(f"🚀 Advanced RAG Response:\n{response}")
        # print("   (Complete the function above to test the full pipeline)")

        print("\n🎯 This should provide:")
        print("   - Filtered relevant results only")
        print("   - Comprehensive analytical response")
        print("   - Combined postprocessing and synthesis")
    else:
        print("❌ Failed to create advanced RAG pipeline")
else:
    print("❌ No index available - run previous cells first")


--- Starting Advanced RAG Pipeline Setup ---
⚙️ Configuring advanced pipeline with similarity_cutoff=0.3 and top_k=10.
✨ Initializing SimilarityPostprocessor with a cutoff of 0.3...
✅ SimilarityPostprocessor created.
✨ Initializing TreeSummarize response synthesizer...
✅ TreeSummarize response synthesizer created.
🧠 Creating RetrieverQueryEngine with SimilarityPostprocessor and TreeSummarize synthesis...
✅ Advanced RAG pipeline (query engine) created.
--- Advanced RAG Pipeline Setup Finished ---

✅ Advanced RAG pipeline created successfully!
   🔧 Similarity filtering: ✅
   🌳 TreeSummarize synthesis: ✅

🔍 Testing complex query: 'Analyze the current state and future potential of AI agent technologies'
🚀 Advanced RAG Response:
AI agent technologies are currently demonstrating significant promise, particularly in their ability to autonomously perform complex tasks through advanced reasoning, planning, and tool execution. These technologies are being implemented using large language models

## 5. Final Test - Compare Basic vs Advanced RAG

Once you've completed all the functions above, run this cell to compare basic RAG with your advanced techniques.


In [ ]:
# Final comparison: Basic vs Advanced RAG
print("🚀 Advanced RAG Techniques Assignment - Final Test")
print("=" * 60)

# Test queries for comparison
test_queries = [
    "What are the key capabilities of AI agents?",
    "How do you evaluate agent performance metrics?",
    "Explain the benefits and challenges of multimodal AI systems"
]

# Check if all components were created
components_status = {
    "Basic Index": index is not None,
    "Similarity Filter": 'filtered_engine' in locals() and filtered_engine is not None,
    "TreeSummarize": 'tree_engine' in locals() and tree_engine is not None,
    "Structured Output": 'structured_program' in locals() and structured_program is not None,
    "Advanced Pipeline": 'advanced_pipeline' in locals() and advanced_pipeline is not None
}

print("\n📊 Component Status:")
for component, status in components_status.items():
    status_icon = "✅" if status else "❌"
    print(f"   {status_icon} {component}")

# Create basic query engine for comparison
if index:
    print("\n🔍 Creating basic query engine for comparison...")
    basic_engine = index.as_query_engine(similarity_top_k=5)

    print("\n" + "=" * 60)
    print("🆚 COMPARISON: Basic vs Advanced RAG")
    print("=" * 60)

    for i, query in enumerate(test_queries, 1):
        print(f"\n📋 Test Query {i}: '{query}'")
        print("-" * 50)

        # Basic RAG
        print("🔹 Basic RAG:")
        if basic_engine:
            # Uncomment when testing:
            # basic_response = basic_engine.query(query)
            # print(f"   Response: {str(basic_response)[:200]}...")
            print("   (Standard vector search + simple response)")

        # Advanced RAG (if implemented)
        print("\n🔸 Advanced RAG:")
        if components_status["Advanced Pipeline"]:
            # Uncomment when testing:
            # advanced_response = advanced_pipeline.query(query)
            # print(f"   Response: {advanced_response}")
            print("   (Filtered + TreeSummarize + Structured output)")
        else:
            print("   Complete the advanced pipeline function to test")

# Final status
print("\n" + "=" * 60)
print("🎯 Assignment Status:")
completed_count = sum(components_status.values())
total_count = len(components_status)

print(f"   Completed: {completed_count}/{total_count} components")

if completed_count == total_count:
    print("\n🎉 Congratulations! You've mastered Advanced RAG Techniques!")
    print("   ✅ Node postprocessors for result filtering")
    print("   ✅ Response synthesizers for better answers")
    print("   ✅ Structured outputs for reliable data")
    print("   ✅ Advanced pipelines combining all techniques")
    print("\n🚀 You're ready for production RAG systems!")
else:
    missing = total_count - completed_count
    print(f"\n📝 Complete {missing} more components to finish the assignment:")
    for component, status in components_status.items():
        if not status:
            print(f"   - {component}")

print("\n💡 Key learnings:")
print("   - Postprocessors improve result relevance and precision")
print("   - Different synthesizers work better for different query types")
print("   - Structured outputs enable reliable system integration")
print("   - Advanced techniques can be combined for production systems")
